In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
import seaborn as sns
import warnings
from sklearn.model_selection import StratifiedKFold,GridSearchCV
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')
import os
os.chdir('E:/Machine_Learning/Cases/Satellite_Imaging/')

In [13]:
satellite = pd.read_csv('Satellite.csv',sep=';')
satellite.head()

,x.1,x.2,x.3,x.4,x.5,x.6,x.7,x.8,x.9,x.10,...,x.28,x.29,x.30,x.31,x.32,x.33,x.34,x.35,x.36,classes
0,92,115,120,94,84,102,106,79,84,102,...,104,88,121,128,100,84,107,113,87,grey soil
1,84,102,106,79,84,102,102,83,80,102,...,100,84,107,113,87,84,99,104,79,grey soil
2,84,102,102,83,80,102,102,79,84,94,...,87,84,99,104,79,84,99,104,79,grey soil
3,80,102,102,79,84,94,102,79,80,94,...,79,84,99,104,79,84,103,104,79,grey soil
4,84,94,102,79,80,94,98,76,80,102,...,79,84,103,104,79,79,107,109,87,grey soil


In [14]:
X ,y = satellite.drop('classes',axis=1),satellite['classes']


In [15]:
clf = IsolationForest(contamination=0.05,random_state=26)
clf.fit(X)
predictions = clf.predict(X)
print("percentage of outliers = "+str((predictions<0).mean()*100)+"%")
ab_ind = np.where(predictions<0)
print('Outliers : ')
print(ab_ind)

percentage of outliers = 5.003885003885004%
Outliers : 
(array([ 133,  134,  188,  204,  205,  206,  250,  251,  267,  268,  269,
        273,  314,  316,  317,  329,  330,  331,  332,  333,  382,  383,
        394,  396,  397,  398,  399,  400,  401,  406,  407,  449,  450,
        463,  464,  465,  466,  467,  468,  469,  470,  474,  475,  512,
        520,  521,  522,  524,  525,  526,  527,  528,  529,  534,  570,
        579,  580,  581,  582,  585,  586,  628,  638,  639,  640,  641,
        642,  644,  649,  650,  651,  652,  661,  703,  704,  706,  707,
        708,  711,  712,  713,  714,  736,  737,  761,  762,  763,  764,
        768,  769,  777,  795,  796,  797,  821,  822,  823,  824,  825,
        828,  829,  857,  878,  879,  880,  881,  882,  883,  884,  891,
        910,  911,  912,  913,  914,  937,  938,  939,  940,  941,  948,
        975,  977,  978,  995,  996,  997,  998,  999, 1004, 1027, 1028,
       1031, 1039, 1040, 1052, 1053, 1054, 1055, 1097, 1098, 1099, 

In [16]:
y[X.index[ab_ind]].value_counts()

classes
cotton crop           263
vegetation stubble     41
grey soil              18
Name: count, dtype: int64

## Training with outliers : 

In [17]:
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
lr = LogisticRegression()
params = {
    'C': np.linspace(0.001,5,10),
    'solver':['lbfgs','newton-cg','newton-cholesky','sag','saga']
}
gcv = GridSearchCV(lr,params,scoring='neg_log_loss',cv =kfold,n_jobs=-1)
gcv.fit(X,y)
print('Best params : ',gcv.best_params_,"Best score : ",gcv.best_score_)

Best params :  {'C': np.float64(0.001), 'solver': 'newton-cholesky'} Best score :  -0.35961804302178474


## Training by removing outliers

In [18]:
sat = satellite[predictions==1]
X ,y = sat.drop('classes',axis=1),sat['classes']
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=26)
lr = LogisticRegression()
params = {
    'C': np.linspace(0.001,5,10),
    'solver':['lbfgs','newton-cg','sag','saga']
}
gcv = GridSearchCV(lr,params,scoring='neg_log_loss',cv =kfold,n_jobs=-1)
gcv.fit(X,y)
print('Best params : ',gcv.best_params_,"Best score : ",gcv.best_score_)

Best params :  {'C': np.float64(0.001), 'solver': 'newton-cg'} Best score :  -0.36799895340665933
